# Homework 10
#### Course Notes
**Language Models:** https://github.com/rjenki/BIOS512/tree/main/lecture17  
**Unix:** https://github.com/rjenki/BIOS512/tree/main/lecture18  
**Docker:** https://github.com/rjenki/BIOS512/tree/main/lecture19

## Question 1
#### Make a language model that uses ngrams and allows the user to specify start words, but uses a random start if one is not specified.

In [2]:
#Load Packages
install.packages("tokenizers")
library(httr)
library(tokenizers)

also installing the dependency ‘SnowballC’





The downloaded binary packages are in
	/var/folders/dx/v8n51kls5l5g188jfm6db16m0000gn/T//RtmpEZPgtv/downloaded_packages


#### a) Make a function to tokenize the text.

In [3]:
tokenize_text <- function(text) {
  tokenizers::tokenize_words(text, lowercase = TRUE, strip_punct = TRUE)[[1]]
}

#### b) Make a function generate keys for ngrams.

In [4]:
key_from <- function(ngram, sep = "\x1f") {
  paste(ngram, collapse = sep)
}

#### c) Make a function to build an ngram table.

In [5]:
build_ngram_table <- function(tokens, n, sep = "\x1f") {
  if (length(tokens) < n) return(new.env(parent = emptyenv()))
  tbl <- new.env(parent = emptyenv())
  for (i in seq_len(length(tokens) - n + 1L)) {
    ngram <- tokens[i:(i + n - 2L)]
    next_word <- tokens[i + n - 1L]
    key <- key_from(ngram, sep)
    
    counts <- if (!is.null(tbl[[key]])) tbl[[key]] else integer(0)
    if (next_word %in% names(counts)) {
      counts[[next_word]] <- counts[[next_word]] + 1L
    } else {
      counts[[next_word]] <- 1L
    }
    tbl[[key]] <- counts
  }
  
  tbl
}

#### d) Function to digest the text.

In [6]:
digest_text <- function(text, n) {
  tokens <- tokenize_text(text)
  build_ngram_table(tokens, n)
}

#### e) Function to digest the url.

In [7]:
digest_url <- function(url, n) {
  res <- httr::GET(url)
  txt <- httr::content(res, as = "text", encoding = "UTF-8")
  digest_text(txt, n)
}

#### f) Function that gives random start.

In [8]:
random_start <- function(tbl, sep = "\x1f") {
  keys <- ls(envir = tbl, all.names = TRUE)
  if (length(keys) == 0) stop("No n-grams available. Digest text first.")
  picked <- sample(keys, 1)
  strsplit(picked, sep, fixed = TRUE)[[1]]
}

#### g) Function to predict the next word.

In [9]:
predict_next_word <- function(tbl, ngram, sep = "\x1f") {
  key <- key_from(ngram, sep)
  counts <- if(!is.null(tbl[[key]])) tbl[[key]] else integer(0)
  if (length(counts) == 0) return(NA_character_)
  sample(names(counts), size = 1, prob = as.numeric(counts))
}

#### h) Function that puts everything together. Specify that if the user does not give a start word, then the random start will be used.

In [10]:
make_ngram_generator <- function(tbl, n, sep = "\x1f") {
  force(tbl); n <- as.integer(n); force(sep)
  
  function(start_words = NULL, length = 10L) {
    if (is.null(start_words) || length(start_words) != n - 1L) {
      start_words <- random_start(tbl, sep)
    }
    
    word_sequence <- start_words
    
    for (i in seq_len(max(0L, length - length(start_words)))) {
      ngram <- tail(word_sequence, n - 1L)
      next_word <- predict_next_word(tbl, ngram, sep)
      if (is.na(next_word)) break
      word_sequence <- c(word_sequence, next_word)
    }
    
    paste(word_sequence, collapse = " ")
  }
}

## Question 2
#### For this question, set `seed=2025`.
#### a) Test your model using a text file of [Grimm's Fairy Tails](https://www.gutenberg.org/cache/epub/2591/pg2591.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

In [12]:
set.seed(2025)
url <- "https://www.gutenberg.org/cache/epub/2591/pg2591.txt"
table <- digest_url(url, n = 3)
generate <- make_ngram_generator(table, n = 3)

In [13]:
output1 <- generate(start_words = c("the", "king"), length = 15)
cat("Output with start words 'the king':\n", output1, "\n\n")

Output with start words 'the king':
 the king has forbidden me to marry another husband am not i shall ride upon 



In [14]:
output2 <- generate(length = 15)
cat("Output with random start:\n", output2, "\n")

Output with random start:
 some when she had given no children so greatly did they would be pope this 


#### b) Test your model using a text file of [Ancient Armour and Weapons in Europe](https://www.gutenberg.org/cache/epub/46342/pg46342.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

In [15]:
set.seed(2025)
url2 <- "https://www.gutenberg.org/files/46342/46342-h/46342-h.htm"
table2 <- digest_url(url2, n = 3)
generate2 <- make_ngram_generator(table2, n = 3)


In [16]:
output_bi <- generate2(start_words = c("the", "king"), length = 15)
cat("b.i) With start words 'the king':\n", output_bi, "\n\n")

b.i) With start words 'the king':
 the king caused his wooden tower of the entombed warrior discovered at leckhampton hill near 



In [1]:
output_bii <- generate2(length = 15)
cat("b.ii) With random start:\n", output_bii, "\n")

ERROR: Error in generate2(length = 15): could not find function "generate2"


#### c) Explain in 1-2 sentences the difference in content generated from each source.

Answer: The content generated from Grimm's Fairy Tales is a narrative, with story-like language and dialogue, whereas the content generated from Ancient Armour and Weapons in Europe has more technical and historical language. The model mimics the style and vocabulary of the source text, so different sources produce very different style of generated text.

## Question 3
#### a) What is a language learning model? 
#### b) Imagine the internet goes down and you can't run to your favorite language model for help. How do you run one locally?

a) Answer: A language model is a type of machine learning model that predicts the next word in a sequence to understand and generate human-like text, like ChatGPT. Most language models provide an HTTP API, which lets programs (not web browsers) send requests to the model and receive responses. 

b) Answer: You can run one locally using OLLAMA to download a language model, which you can run without needing an online API. OLLAMA uses Docker to create a lightweight local environment, and once installed, you can interact with the model through HTTP requests using R packages like httr to run a langage model locally.

## Question 4
#### Explain what the following vocab words mean in the context of typing `mkdir project` into the command line. If the term doesn't apply to this command, give the definition and/or an example.
| Term | Meaning |  
|------|---------|
| **Shell** | The shell interprets the command mkdir project and runs the mkdir program.  |
| **Terminal emulator** |The terminal emulator is the window you type the command into, which displays the shell.  |
| **Process** |Running the command creates a new process for mkdir  |
| **Signal** | A signal is a message sent to a process to tell it what to do.|
| **Standard input** |Standard input is the stream a process reads from, usually the keyboard.  |
| **Standard output** |Standard output is the stream where a process prints normal messages.  |
| **Command line argument** | Project is the command line argument passed to mkdir.  |
| **The environment** | The environment is a set of variables the shell passes to the mkdir process when it starts. For mkdir project, it includes things like PATH, which tells the shell where to find the mkdir program. |

## Question 5
#### Consider the following command `find . -iname "*.R" | xargs grep read_csv`.
#### a) What are the programs?
#### b) Explain what this command is doing, part by part.

a) Answer: The programs are find, xargs, and grep  

b)Answer: find . -iname "*.R", which tells the find program to look in the current directory and all of its subdirectories for any files whose names end in .R, ignoring case. The pipe symbol then takes the list of matching filenames produced by find and passes them as input to the next program. The xargs grep read_csv portion receives that list of filenames and uses xargs to supply them as arguments to the grep program. Finally, grep read_csv searches inside each of those .R files for the text string read_csv and prints any lines where it occurs.

## Question 6
#### Install Docker on your machine. See [here](https://github.com/rjenki/BIOS512/blob/main/lecture18/docker_install.md) for instructions. 
#### a) Show the response when you run `docker run hello-world`.
#### b) Access Rstudio through a Docker container. Set your password and make sure your files show up on the Rstudio server. Type the command and the output you get below.
#### c) How do you log in to the RStudio server?

a) Answer: 

Hello from Docker!  
This message shows that your installation appears to be working correctly.  

To generate this message, Docker took the following steps:  
 1. The Docker client contacted the Docker daemon.  
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.  
    (amd64) 
 3. The Docker daemon created a new container from that image which runs the  
    executable that produces the output you are currently reading.  
 4. The Docker daemon streamed that output to the Docker client, which sent it  
    to your terminal.  

To try something more ambitious, you can run an Ubuntu container with:  
 $ docker run -it ubuntu bash  

Share images, automate workflows, and more with a free Docker ID:  
 https://hub.docker.com/  

For more examples and ideas, visit:  
 https://docs.docker.com/get-started/  

b) Commands: 
docker pull rocker/rstudio
docker run -d -p 8788:8787 -v ~/Desktop/BIOS512:/home/rstudio/files -e PASSWORD=bios512dockerpassword rocker/rstudio

Output:

Unable to find image 'rocker/rstudio:latest' locally  
latest: Pulling from rocker/rstudio  
3665120d345d: Download complete   
3665120d345d: Pull complete   
4b3ffd8ccb52: Pull complete   
664fb1818bbb: Pull complete   
890065c4c99d: Download complete   
890065c4c99d: Pull complete   
2c9ba66d5dbe: Pull complete   
39038e16d1ba: Pull complete   
971ba7cf0d8a: Download complete   
971ba7cf0d8a: Pull complete   
d923cf803a12: Pull complete   
e4b9e87bb831: Pull complete     
62f215ca34c6: Pull complete   
2a63ed8b2250: Pull complete   
9c1a4a0706b7: Pull complete   
5d246ec925db: Pull complete   
b71e78fefbbb: Pull complete   
3c7cdccc4be7: Pull complete   
docker pull rocker/rstudio  
Digest: sha256:9f85211a666fb426081a6f5a01f9f9f51655262258419fa21e0ce38a5afc78d8  
Status: Downloaded newer image for rocker/rstudio:latest  
2c61580641e282149a2d37f51b2c5e2fed74b2bead75e23e5fd514e1498ec2e9  

c) Answer: To log in to an RStudio Server running in Docker, first go to http://localhost:8787. The username is rstudio, and the password is the one set when starting the container. After logging in, the Files tab in RStudio allows you to access and work with your files directly in the RStudio environment.